# 01_bootstrap_alpaca

Fetches 5 years of daily OHLCV history (2020-01-01 to today) for every
ticker in `precursor.bronze.universe` and writes to
`precursor.bronze.alpaca_raw`.

**Run this notebook ONCE manually.** After bootstrap, `02_ingestion.py`
handles daily updates.

Resume-safe: a checkpoint table (`precursor.bronze.bootstrap_checkpoint`)
tracks completed (ticker, year) pairs so re-runs skip finished work.

In [0]:
%pip install alpaca-py websockets msgpack --no-deps

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# We must pin numpy<2.0 separately because alpaca-py's transitive deps can pull
# in numpy 2.x, which breaks pandas' compiled C extensions (ABI changed in 2.0).
%pip install "numpy<2.0" --no-deps

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install "pydantic>=2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 57.3 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.10.0
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.11/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-3ecc8788-43e3-4fd6-9868-1c3ff400f2ce
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
  Attempting uninstall: pydantic
    Found existing installation: pydantic 1.10.6
    Not uninstalling pydantic at /databricks/python3/lib/python3.11/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-3ecc8788-43e3-4fd6-9868-1c3ff400f2ce
    Can't uninstall 'pydantic'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
alpaca-py 0.43.4 requires msgpack<2.0.0,>=1.0.3, w

In [0]:
dbutils.library.restartPython()

In [0]:
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.data.enums import Adjustment

import pandas as pd
import logging
from collections import defaultdict
from datetime import datetime, date, timedelta
from typing import Optional

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DateType,
    DoubleType, LongType, TimestampType, BooleanType,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("precursor.alpaca_bootstrap")

START_DATE = "2020-01-01"
END_DATE   = datetime.today().strftime("%Y-%m-%d")
BATCH_SIZE = 50  # tickers per Alpaca API call

# All years that may contain data between START_DATE and today.
# We chunk by year to stay well under the 2.5-hour serverless timeout.
YEARS = [2020, 2021, 2022, 2023, 2024, 2025, 2026]

logger.info("START_DATE=%s  END_DATE=%s  BATCH_SIZE=%d", START_DATE, END_DATE, BATCH_SIZE)

INFO:precursor.alpaca_bootstrap:START_DATE=2020-01-01  END_DATE=2026-05-01  BATCH_SIZE=50


## Cell 3 — Load credentials and universe

In [0]:
def get_alpaca_client() -> StockHistoricalDataClient:
    """Build an Alpaca StockHistoricalDataClient from Databricks secrets.

    Secrets are stored in scope="precursor" under keys "alpaca-key" and
    "alpaca-secret". Never hardcode credentials.

    Returns:
        Configured StockHistoricalDataClient.
    """
    api_key    = dbutils.secrets.get(scope="precursor", key="alpaca-key")
    api_secret = dbutils.secrets.get(scope="precursor", key="alpaca-secret")
    return StockHistoricalDataClient(api_key=api_key, secret_key=api_secret)


def load_universe_tickers() -> list[str]:
    """Load the full set of tickers from precursor.bronze.universe.

    We fetch ALL distinct tickers that were ever in the index (in_index = TRUE
    at any point), including those later removed.

    Survivorship-bias note: fetching only currently-active tickers would
    introduce survivorship bias — the model would never see data for companies
    that were removed (e.g., due to bankruptcy or acquisition), making the
    training set look artificially healthy.  We therefore include all
    historical members, even if they have no recent price data.

    Returns:
        Sorted list of ticker strings.
    """
    df = spark.sql("""
        SELECT DISTINCT ticker
        FROM precursor.bronze.universe
        WHERE in_index = TRUE
        ORDER BY ticker
    """).toPandas()

    tickers = df["ticker"].tolist()
    logger.info("Universe tickers loaded: %d total", len(tickers))
    return tickers

## Cell 4 — Checkpoint management

In [0]:
_CHECKPOINT_TABLE = "precursor.bronze.bootstrap_checkpoint"

_CHECKPOINT_SCHEMA = StructType([
    StructField("ticker",   StringType(),    nullable=False),
    StructField("year",     IntegerType(),   nullable=False),
    StructField("status",   StringType(),    nullable=False),
    StructField("saved_at", TimestampType(), nullable=False),
])


def get_completed_years(ticker: str) -> list[int]:
    """Return years already successfully completed for a ticker.

    Reads the checkpoint table if it exists; returns an empty list if the
    table does not yet exist (first run).

    Args:
        ticker: Ticker symbol.

    Returns:
        List of integer years that have status='completed' for this ticker.
    """
    try:
        result = spark.sql(f"""
            SELECT year
            FROM {_CHECKPOINT_TABLE}
            WHERE ticker = '{ticker}'
              AND status = 'completed'
        """).toPandas()
        return result["year"].tolist()
    except Exception:
        # Table does not exist yet on the very first run
        return []


def save_checkpoint(ticker: str, year: int, status: str) -> None:
    """Upsert a (ticker, year) checkpoint record.

    Appends a new row then deduplicates on (ticker, year), keeping the most
    recent entry.  This is the simplest safe approach for serverless
    (no MERGE INTO needed, no streaming).

    Args:
        ticker: Ticker symbol.
        year:   Calendar year of the data batch.
        status: "completed" or "failed".
    """
    row = pd.DataFrame([{
        "ticker":   ticker,
        "year":     year,
        "status":   status,
        "saved_at": datetime.utcnow(),
    }])

    spark_row = spark.createDataFrame(row, schema=_CHECKPOINT_SCHEMA)

    # Append the new record
    (
        spark_row.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(_CHECKPOINT_TABLE)
    )

def get_remaining_work(
    all_tickers: list[str],
    years: list[int],
) -> list[tuple[str, int]]:
    """Return (ticker, year) pairs not yet checkpointed as completed.

    Allows safe resume after a timeout without re-fetching finished work.

    Args:
        all_tickers: Full universe ticker list.
        years:       List of calendar years to process.

    Returns:
        List of (ticker, year) tuples that still need fetching.
    """
    try:
        done_df = spark.sql(f"""
            SELECT ticker, year
            FROM {_CHECKPOINT_TABLE}
            WHERE status = 'completed'
        """).toPandas()
        done_set = set(zip(done_df["ticker"], done_df["year"]))
    except Exception:
        done_set = set()

    remaining = [
        (t, y)
        for t in all_tickers
        for y in years
        if (t, y) not in done_set
    ]
    logger.info(
        "Remaining work: %d ticker-year pairs (%d already completed)",
        len(remaining),
        len(all_tickers) * len(years) - len(remaining),
    )
    return remaining

## Cell 5 — Fetch OHLCV for one ticker, one year

In [0]:
def fetch_ticker_year(
    client: StockHistoricalDataClient,
    ticker: str,
    year: int,
) -> Optional[pd.DataFrame]:
    """Fetch daily OHLCV bars for a single ticker and calendar year.

    Uses Adjustment.ALL so that prices are adjusted for both splits AND
    dividends.  This is non-negotiable: without split adjustment the model
    would see apparent price crashes on split dates that are not real price
    moves.  Without dividend adjustment, total-return comparisons across
    tickers would be distorted.

    Args:
        client: Authenticated Alpaca StockHistoricalDataClient.
        ticker: Ticker symbol (e.g. "AAPL").
        year:   Calendar year (e.g. 2022).

    Returns:
        DataFrame with columns
            ticker, date, open, high, low, close, volume, vwap, trade_count,
        or None if no data was returned (delisted / pre-IPO ticker).
    """
    start = max(f"{year}-01-01", START_DATE)
    end   = min(f"{year}-12-31", END_DATE)

    # Nothing to fetch if the year is entirely outside our window
    if start > end:
        return None

    try:
        request = StockBarsRequest(
            symbol_or_symbols=[ticker],
            timeframe=TimeFrame.Day,
            start=start,
            end=end,
            adjustment=Adjustment.ALL,  # split + dividend adjusted — see docstring
        )
        bars = client.get_stock_bars(request)
        df   = bars.df
    except Exception as exc:
        logger.warning("%s %d: API error — %s", ticker, year, exc)
        return None

    if df is None or df.empty:
        logger.warning("%s %d: no data returned (possibly delisted)", ticker, year)
        return None

    # bars.df has a MultiIndex (symbol, timestamp); reset to flat
    df = df.reset_index()

    # Rename columns to canonical schema
    rename_map = {
        "symbol":      "ticker",
        "timestamp":   "date",
        "open":        "open",
        "high":        "high",
        "low":         "low",
        "close":       "close",
        "volume":      "volume",
        "vwap":        "vwap",
        "trade_count": "trade_count",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    # If symbol column was named differently, ensure ticker column exists
    if "ticker" not in df.columns:
        df["ticker"] = ticker

    # Strip timezone so Spark DateType() serialisation does not complain
    if pd.api.types.is_datetime64tz_dtype(df["date"]):
        df["date"] = df["date"].dt.tz_localize(None)

    df["date"] = pd.to_datetime(df["date"]).dt.date

    # Keep only the columns we need
    keep = ["ticker", "date", "open", "high", "low", "close", "volume", "vwap", "trade_count"]
    df = df[[c for c in keep if c in df.columns]]

    # Fill missing optional columns with None
    for col in ("vwap", "trade_count"):
        if col not in df.columns:
            df[col] = None

    return df

## Cell 6 — Batch fetch and write one year

In [0]:
_ALPACA_RAW_SCHEMA = StructType([
    StructField("ticker",      StringType(),    nullable=False),
    StructField("date",        DateType(),      nullable=False),
    StructField("open",        DoubleType(),    nullable=True),
    StructField("high",        DoubleType(),    nullable=True),
    StructField("low",         DoubleType(),    nullable=True),
    StructField("close",       DoubleType(),    nullable=True),
    StructField("volume",      LongType(),      nullable=True),
    StructField("vwap",        DoubleType(),    nullable=True),
    StructField("trade_count", LongType(),      nullable=True),
    StructField("adjusted",    BooleanType(),   nullable=False),
    StructField("ingested_at", TimestampType(), nullable=False),
])


def _pd_batch_to_spark(batch_frames: list[pd.DataFrame]) -> Optional[DataFrame]:
    """Concatenate a list of pandas DataFrames and convert to a Spark DataFrame.

    Args:
        batch_frames: Non-empty list of pandas DataFrames with OHLCV columns.

    Returns:
        Spark DataFrame matching _ALPACA_RAW_SCHEMA, or None if list is empty.
    """
    if not batch_frames:
        return None

    combined = pd.concat(batch_frames, ignore_index=True)
    combined["adjusted"]    = True
    combined["ingested_at"] = datetime.utcnow()

    # Ensure numeric types match schema — Alpaca sometimes returns floats for volume
    combined["volume"]      = combined["volume"].astype("Int64").where(combined["volume"].notna(), None)
    combined["trade_count"] = combined["trade_count"].astype("Int64").where(combined["trade_count"].notna(), None)

    # Convert pandas nullable Int64 to plain Python int so Spark is happy
    combined["volume"]      = combined["volume"].apply(lambda x: int(x) if pd.notna(x) else None)
    combined["trade_count"] = combined["trade_count"].apply(lambda x: int(x) if pd.notna(x) else None)

    # date must be datetime64 (not python date) for Spark DateType
    combined["date"] = pd.to_datetime(combined["date"])

    return spark.createDataFrame(combined, schema=_ALPACA_RAW_SCHEMA)


def fetch_and_write_year(
    client: StockHistoricalDataClient,
    tickers: list[str],
    year: int,
) -> dict[str, int]:
    """Fetch all tickers for one calendar year and upsert into alpaca_raw.

    Processes tickers in batches of BATCH_SIZE.  After each batch the new
    rows are unioned with existing data for that year, deduplicated on
    (ticker, date), and the year partition is overwritten.  A checkpoint is
    written for every successfully fetched ticker.

    Args:
        client:  Authenticated Alpaca client.
        tickers: List of ticker symbols to fetch for this year.
        year:    Calendar year.

    Returns:
        Dict mapping ticker → number of rows fetched (0 for tickers with no data).
    """
    results: dict[str, int] = {}
    batches  = [tickers[i:i + BATCH_SIZE] for i in range(0, len(tickers), BATCH_SIZE)]
    total_b  = len(batches)
    now_str  = lambda: datetime.utcnow().isoformat(timespec="seconds")

    for batch_idx, batch in enumerate(batches, start=1):
        batch_frames: list[pd.DataFrame] = []

        for ticker in batch:
            df = fetch_ticker_year(client, ticker, year)
            if df is not None and not df.empty:
                batch_frames.append(df)
                results[ticker] = len(df)
            else:
                results[ticker] = 0

        row_count = sum(len(f) for f in batch_frames)
        logger.info(
            "Batch %d/%d  year=%d  tickers=%d  rows=%d  [%s]",
            batch_idx, total_b, year, len(batch), row_count, now_str(),
        )

        if not batch_frames:
            now = datetime.utcnow()
            ckpt_df = pd.DataFrame([
                {"ticker": t, "year": year, "status": "completed", "saved_at": now}
                for t in batch
            ])
            (
                spark.createDataFrame(ckpt_df, schema=_CHECKPOINT_SCHEMA)
                .write.format("delta").mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(_CHECKPOINT_TABLE)
            )
            continue

        new_spark = _pd_batch_to_spark(batch_frames)
        if new_spark is None:
            continue

        # Upsert: union new rows with existing data for this year, then deduplicate
        try:
            existing = spark.sql(f"""
                SELECT * FROM precursor.bronze.alpaca_raw
                WHERE YEAR(date) = {year}
                  AND ticker IN ({",".join(f"'{t}'" for t in batch)})
            """)
            combined_spark = existing.union(new_spark)
        except Exception:
            # Table does not exist yet on the first batch of the first year
            combined_spark = new_spark

        # Keep latest ingested_at per (ticker, date)
        deduped = (
            combined_spark
            .withColumn(
                "rn",
                F.row_number().over(
                    Window.partitionBy("ticker", "date").orderBy(F.desc("ingested_at"))
                )
            )
            .filter(F.col("rn") == 1)
            .drop("rn")
        )

        (
            deduped.write
            .format("delta")
            .mode("overwrite")
            .option("replaceWhere", f"YEAR(date) = {year} AND ticker IN ({','.join(repr(t) for t in batch)})")
            .option("mergeSchema", "true")
            .saveAsTable("precursor.bronze.alpaca_raw")
        )

        # Checkpoint all tickers in one write
        now = datetime.utcnow()
        ckpt_df = pd.DataFrame([
            {"ticker": t, "year": year, "status": "completed", "saved_at": now}
            for t in batch
        ])
        (
            spark.createDataFrame(ckpt_df, schema=_CHECKPOINT_SCHEMA)
            .write.format("delta").mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(_CHECKPOINT_TABLE)
        )

    return results

## Cell 7 — Flag delisted tickers

In [0]:
def flag_delisted_tickers(
    tickers: list[str],
    results: dict[str, int],
) -> pd.DataFrame:
    """Identify tickers that returned zero rows across all years and log them.

    A ticker with zero rows for every year is likely fully delisted before
    our window began, or was never listed on a US exchange supported by
    Alpaca.  We write these to a separate table so downstream processes can
    exclude them without performing expensive joins against alpaca_raw.

    Args:
        tickers: Full universe ticker list that was attempted.
        results: Dict of ticker → total rows fetched (summed across all years).

    Returns:
        Pandas DataFrame of delisted tickers that was written.
    """
    delisted = [t for t in tickers if results.get(t, 0) == 0]

    if not delisted:
        logger.info("No fully-delisted tickers found (all tickers had at least some data).")
        return pd.DataFrame(columns=["ticker", "reason", "flagged_at"])

    logger.info("Delisted / no-data tickers found: %d", len(delisted))

    df = pd.DataFrame({
        "ticker":     delisted,
        "reason":     "no data returned from Alpaca",
        "flagged_at": datetime.utcnow(),
    })

    schema = StructType([
        StructField("ticker",     StringType(),    nullable=False),
        StructField("reason",     StringType(),    nullable=True),
        StructField("flagged_at", TimestampType(), nullable=False),
    ])

    spark_df = spark.createDataFrame(df, schema=schema)
    (
        spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("precursor.bronze.delisted_tickers")
    )

    logger.info("precursor.bronze.delisted_tickers written: %d tickers.", len(delisted))
    return df

## Cell 8 — Main execution

In [0]:
_bootstrap_start = datetime.now()
logger.info(
    "=== precursor.01_bootstrap_alpaca START at %s ===",
    _bootstrap_start.isoformat(),
)

all_results: dict[str, int] = {}  # ticker → total rows across all years

try:
    client  = get_alpaca_client()
    tickers = load_universe_tickers()

    remaining = get_remaining_work(tickers, YEARS)
    logger.info("Starting bootstrap for %d ticker-year pairs", len(remaining))

    # Build a per-year lookup so we only pass tickers that still need work
    tickers_by_year: dict[int, list[str]] = defaultdict(list)
    for t, y in remaining:
        tickers_by_year[y].append(t)

    for year in YEARS:
        year_tickers = tickers_by_year.get(year, [])
        if not year_tickers:
            logger.info("Year %d — all tickers already completed, skipping.", year)
            continue

        year_start = datetime.now()
        logger.info("Processing year %d  (%d tickers remaining)", year, len(year_tickers))

        year_results = fetch_and_write_year(client, year_tickers, year)

        # Accumulate totals
        for t, n in year_results.items():
            all_results[t] = all_results.get(t, 0) + n

        elapsed_min = (datetime.now() - year_start).total_seconds() / 60
        logger.info("Year %d complete in %.2f min.", year, elapsed_min)

    delisted_df = flag_delisted_tickers(tickers, all_results)

    _total_elapsed = (datetime.now() - _bootstrap_start).total_seconds() / 60
    logger.info(
        "=== precursor.01_bootstrap_alpaca END — %.2f min total ===",
        _total_elapsed,
    )

    tickers_with_data    = sum(1 for n in all_results.values() if n > 0)
    tickers_without_data = sum(1 for n in all_results.values() if n == 0)
    total_rows           = sum(all_results.values())

    print("=" * 60)
    print("  PRECURSOR — ALPACA BOOTSTRAP SUMMARY")
    print("=" * 60)
    print(f"  Total tickers attempted       : {len(tickers)}")
    print(f"  Tickers with data             : {tickers_with_data}")
    print(f"  Tickers with no data          : {tickers_without_data}  (delisted / unsupported)")
    print(f"  Total rows written            : {total_rows:,}")
    print(f"  Years processed               : {sorted(tickers_by_year.keys())}")
    print(f"  Elapsed time                  : {_total_elapsed:.2f} min")
    print("=" * 60)

except Exception as exc:
    logger.error("Bootstrap failed: %s", exc, exc_info=True)
    logger.error("Resume by re-running — checkpoint will skip completed work.")
    # Do not re-raise: let the notebook cell finish so the checkpoint data is preserved

INFO:precursor.alpaca_bootstrap:=== precursor.01_bootstrap_alpaca START at 2026-05-01T13:24:54.626039 ===
INFO:precursor.alpaca_bootstrap:Universe tickers loaded: 613 total
INFO:precursor.alpaca_bootstrap:Remaining work: 3880 ticker-year pairs (411 already completed)
INFO:precursor.alpaca_bootstrap:Starting bootstrap for 3880 ticker-year pairs
INFO:precursor.alpaca_bootstrap:Processing year 2020  (202 tickers remaining)
INFO:precursor.alpaca_bootstrap:Batch 1/5  year=2020  tickers=50  rows=11844  [2026-05-01T13:24:59]
INFO:precursor.alpaca_bootstrap:Batch 2/5  year=2020  tickers=50  rows=11404  [2026-05-01T13:25:08]
INFO:precursor.alpaca_bootstrap:Batch 3/5  year=2020  tickers=50  rows=12600  [2026-05-01T13:25:15]
INFO:precursor.alpaca_bootstrap:Batch 4/5  year=2020  tickers=50  rows=11445  [2026-05-01T13:25:22]
INFO:precursor.alpaca_bootstrap:Batch 5/5  year=2020  tickers=2  rows=504  [2026-05-01T13:25:26]
INFO:precursor.alpaca_bootstrap:Year 2020 complete in 0.56 min.
INFO:precursor.

  PRECURSOR — ALPACA BOOTSTRAP SUMMARY
  Total tickers attempted       : 613
  Tickers with data             : 608
  Tickers with no data          : 5  (delisted / unsupported)
  Total rows written            : 815,168
  Years processed               : [2020, 2021, 2022, 2023, 2024, 2025, 2026]
  Elapsed time                  : 21.03 min


## Cell 9 — Validation

In [0]:
logger.info("=== Running post-bootstrap validation ===")

_checks: list[tuple[str, bool, str]] = []

def _check(name: str, passed: bool, detail: str = "") -> None:
    status = "PASS" if passed else "FAIL"
    msg    = f"[{status}] {name}" + (f" — {detail}" if detail else "")
    logger.info(msg)
    _checks.append((name, passed, detail))


try:
    row_count = spark.table("precursor.bronze.alpaca_raw").count()
    _check(
        "Row count > 500,000",
        row_count > 500_000,
        f"{row_count:,} rows  (expected ~765k for 613 tickers × ~250 days × 5 years)",
    )

    latest_date_row = spark.sql(
        "SELECT MAX(date) AS max_date FROM precursor.bronze.alpaca_raw"
    ).collect()[0]
    latest_date = latest_date_row["max_date"]
    yesterday = (datetime.today() - timedelta(days=1)).date()
    _check(
        "Latest date is yesterday or today",
        latest_date >= yesterday,
        f"latest date = {latest_date}",
    )

    null_close = spark.sql(
        "SELECT COUNT(*) AS n FROM precursor.bronze.alpaca_raw WHERE close IS NULL"
    ).collect()[0]["n"]
    _check("No null close prices", null_close == 0, f"{null_close} nulls found")

    not_adjusted = spark.sql(
        "SELECT COUNT(*) AS n FROM precursor.bronze.alpaca_raw WHERE adjusted != TRUE"
    ).collect()[0]["n"]
    _check("adjusted = TRUE for all rows", not_adjusted == 0, f"{not_adjusted} non-adjusted rows")

    distinct_tickers = spark.sql(
        "SELECT COUNT(DISTINCT ticker) AS n FROM precursor.bronze.alpaca_raw"
    ).collect()[0]["n"]
    _check(
        "At least 500 distinct tickers",
        distinct_tickers >= 500,
        f"{distinct_tickers} distinct tickers",
    )

    min_year, max_year = spark.sql(
        "SELECT MIN(YEAR(date)) AS min_y, MAX(YEAR(date)) AS max_y FROM precursor.bronze.alpaca_raw"
    ).collect()[0]
    _check(
        "Date range spans 2020 to current year",
        min_year <= 2020 and max_year >= datetime.today().year,
        f"years {min_year}–{max_year}",
    )

except Exception as exc:
    logger.error("Validation query failed: %s", exc, exc_info=True)
    _check("Validation queries executed without error", False, str(exc))

passed_n = sum(1 for _, p, _ in _checks if p)
failed_n = len(_checks) - passed_n

print("=" * 60)
print("  VALIDATION RESULTS")
print("=" * 60)
for name, passed, detail in _checks:
    status = "PASS" if passed else "FAIL"
    line   = f"  [{status}] {name}"
    if detail:
        line += f"\n         {detail}"
    print(line)
print("-" * 60)
print(f"  {passed_n} passed  /  {failed_n} failed")
if failed_n > 0:
    print("  WARNING: validation failures detected — review logs above.")
print("=" * 60)

INFO:precursor.alpaca_bootstrap:=== Running post-bootstrap validation ===
INFO:precursor.alpaca_bootstrap:[PASS] Row count > 500,000 — 917,749 rows  (expected ~765k for 613 tickers × ~250 days × 5 years)
INFO:precursor.alpaca_bootstrap:[PASS] Latest date is yesterday or today — latest date = 2026-04-30
INFO:precursor.alpaca_bootstrap:[PASS] No null close prices — 0 nulls found
INFO:precursor.alpaca_bootstrap:[PASS] adjusted = TRUE for all rows — 0 non-adjusted rows
INFO:precursor.alpaca_bootstrap:[PASS] At least 500 distinct tickers — 613 distinct tickers
INFO:precursor.alpaca_bootstrap:[PASS] Date range spans 2020 to current year — years 2020–2026


  VALIDATION RESULTS
  [PASS] Row count > 500,000
         917,749 rows  (expected ~765k for 613 tickers × ~250 days × 5 years)
  [PASS] Latest date is yesterday or today
         latest date = 2026-04-30
  [PASS] No null close prices
         0 nulls found
  [PASS] adjusted = TRUE for all rows
         0 non-adjusted rows
  [PASS] At least 500 distinct tickers
         613 distinct tickers
  [PASS] Date range spans 2020 to current year
         years 2020–2026
------------------------------------------------------------
  6 passed  /  0 failed


In [0]:
# Alpaca rejects "BF/B" and "BRK/B" — their API uses dot notation instead.
# We fetch using the dot form but store using the slash form to match the universe table.
_SLASH_TICKER_MAP = {"BF/B": "BF.B", "BRK/B": "BRK.B"}

_patch_client = get_alpaca_client()
_patch_frames = []

for _universe_ticker, _alpaca_ticker in _SLASH_TICKER_MAP.items():
    logger.info("Patching %s (fetching as %s)", _universe_ticker, _alpaca_ticker)
    for _year in YEARS:
        _start = max(f"{_year}-01-01", START_DATE)
        _end   = min(f"{_year}-12-31", END_DATE)
        if _start > _end:
            continue
        try:
            _request = StockBarsRequest(
                symbol_or_symbols=[_alpaca_ticker],
                timeframe=TimeFrame.Day,
                start=_start,
                end=_end,
                adjustment=Adjustment.ALL,
            )
            _bars = _patch_client.get_stock_bars(_request)
            _df   = _bars.df
        except Exception as _exc:
            logger.warning("%s %d: API error — %s", _alpaca_ticker, _year, _exc)
            continue

        if _df is None or _df.empty:
            logger.warning("%s %d: no data", _alpaca_ticker, _year)
            continue

        _df = _df.reset_index()
        _df = _df.rename(columns={"symbol": "ticker", "timestamp": "date", "trade_count": "trade_count"})
        _df["ticker"] = _universe_ticker  # store as slash form
        if pd.api.types.is_datetime64tz_dtype(_df["date"]):
            _df["date"] = _df["date"].dt.tz_localize(None)
        _df["date"] = pd.to_datetime(_df["date"]).dt.date
        for _col in ("vwap", "trade_count"):
            if _col not in _df.columns:
                _df[_col] = None
        _df["adjusted"]    = True
        _df["ingested_at"] = datetime.utcnow()
        _df["volume"]      = _df["volume"].apply(lambda x: int(x) if pd.notna(x) else None)
        _df["trade_count"] = _df["trade_count"].apply(lambda x: int(x) if pd.notna(x) else None)
        _df["date"]        = pd.to_datetime(_df["date"])
        _patch_frames.append(_df[["ticker","date","open","high","low","close","volume","vwap","trade_count","adjusted","ingested_at"]])
        logger.info("  %s %d: %d rows", _universe_ticker, _year, len(_df))

if _patch_frames:
    _patch_combined = pd.concat(_patch_frames, ignore_index=True)
    _patch_spark    = spark.createDataFrame(_patch_combined, schema=_ALPACA_RAW_SCHEMA)
    _tickers_sql    = ",".join(f"'{t}'" for t in _SLASH_TICKER_MAP)

    try:
        _existing = spark.sql(f"SELECT * FROM precursor.bronze.alpaca_raw WHERE ticker IN ({_tickers_sql})")
        _patch_spark = _existing.union(_patch_spark)
    except Exception:
        pass

    _patch_deduped = (
        _patch_spark
        .withColumn("rn", F.row_number().over(Window.partitionBy("ticker", "date").orderBy(F.desc("ingested_at"))))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )
    (
        _patch_deduped.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"ticker IN ({_tickers_sql})")
        .option("mergeSchema", "true")
        .saveAsTable("precursor.bronze.alpaca_raw")
    )
    logger.info("Patch complete: %d rows written for %s", len(_patch_combined), list(_SLASH_TICKER_MAP))

    for _t in _SLASH_TICKER_MAP:
        _r = spark.sql(f"SELECT COUNT(*) AS n, MIN(date) AS mn, MAX(date) AS mx FROM precursor.bronze.alpaca_raw WHERE ticker = '{_t}'").collect()[0]
        print(f"  {_t}: {_r['n']} rows  {_r['mn']} → {_r['mx']}")
else:
    logger.warning("No data fetched for slash tickers — check Alpaca credentials or symbol availability.")


INFO:precursor.alpaca_bootstrap:Patching BF/B (fetching as BF.B)
INFO:precursor.alpaca_bootstrap:  BF/B 2020: 252 rows
INFO:precursor.alpaca_bootstrap:  BF/B 2021: 251 rows
INFO:precursor.alpaca_bootstrap:  BF/B 2022: 251 rows
INFO:precursor.alpaca_bootstrap:  BF/B 2023: 250 rows
INFO:precursor.alpaca_bootstrap:  BF/B 2024: 251 rows
INFO:precursor.alpaca_bootstrap:  BF/B 2025: 249 rows
INFO:precursor.alpaca_bootstrap:  BF/B 2026: 82 rows
INFO:precursor.alpaca_bootstrap:Patching BRK/B (fetching as BRK.B)
INFO:precursor.alpaca_bootstrap:  BRK/B 2020: 252 rows
INFO:precursor.alpaca_bootstrap:  BRK/B 2021: 251 rows
INFO:precursor.alpaca_bootstrap:  BRK/B 2022: 251 rows
INFO:precursor.alpaca_bootstrap:  BRK/B 2023: 250 rows
INFO:precursor.alpaca_bootstrap:  BRK/B 2024: 251 rows
INFO:precursor.alpaca_bootstrap:  BRK/B 2025: 249 rows
INFO:precursor.alpaca_bootstrap:  BRK/B 2026: 82 rows
INFO:precursor.alpaca_bootstrap:Patch complete: 3172 rows written for ['BF/B', 'BRK/B']


  BF/B: 1586 rows  2020-01-02 → 2026-04-30
  BRK/B: 1586 rows  2020-01-02 → 2026-04-30
